In [ ]:
from functools import partial

import jax
import jax.numpy as jnp
import optax
from flax import nnx

from utils import HamiltonianOperator

### JAX impl of the NN ansatz


In [9]:
# generic MLP helper class
class MLP(nnx.Module):
    def __init__(
        self,
        hidden_layers: int,
        input_dim: int,
        dim_feedforward: int,
        output_dim: int,
        rngs: nnx.Rngs,
        activation=nnx.gelu,
    ):
        # make it
        net = [
            nnx.Linear(input_dim, dim_feedforward, rngs=rngs),
            activation,
        ]
        for _ in range(hidden_layers):
            net.append(nnx.Linear(dim_feedforward, dim_feedforward, rngs=rngs))
            net.append(activation)
        net.append(nnx.Linear(dim_feedforward, output_dim, rngs=rngs))
        self.MLP = nnx.Sequential(*net)

    def __call__(self, x: jax.Array):
        return self.MLP(x)


# quick helper function
def view_as_complex(x: jax.Array):
    return x[..., 0] + (1j * x[..., 1])


class psi_2D_spinful(nnx.Module):
    def __init__(
        self,
        L: int,
        hidden_layers: int,
        dim_feedforward: int,
        rngs: nnx.Rngs,
        activation=nnx.gelu,
    ):
        self.L = L
        # weights, drawn from a Gaussian distribution (for now)
        self.w_s = nnx.Param(rngs.normal((2 * (L**2),)))
        # instantiate MLPs
        self.f = MLP(hidden_layers, 5, dim_feedforward, 2, rngs, activation)
        self.g_1_prime = MLP(hidden_layers, 1, dim_feedforward, 2, rngs, activation)
        self.g_2_prime = MLP(hidden_layers, 1, dim_feedforward, 2, rngs, activation)

    # antisymmetric factor $F: \widetilde{\Lambda}^n \rightarrow \mathbb{C}$
    def F_antisymmetric(self, x: jax.Array):
        # x should be integers of shape (batch_size, n, 3) with entries from \widetilde{\Lambda}
        # contains (x, y, \sigma)
        # convert spatial coords to complex numbers e^{2\pi i x/L}
        z_x = jnp.stack(
            [
                jnp.cos((2 * jnp.pi * x[:, :, 0]) / self.L),
                jnp.sin((2 * jnp.pi * x[:, :, 0]) / self.L),
                jnp.cos((2 * jnp.pi * x[:, :, 1]) / self.L),
                jnp.sin((2 * jnp.pi * x[:, :, 1]) / self.L),
                (2 * x[:, :, 2]) - 1,  # encode spin as ±1
            ],
            axis=-1,
        )  # should be (batch_size, n, 5) now
        # pass through f MLP
        f_x = view_as_complex(self.f(z_x))  # complex (batch_size, n)
        vandermonde_matrix = jax.vmap(partial(jnp.vander, increasing=True))(
            f_x
        )  # should be of shape (batch_size, n, n)
        # take the determinant
        sign, logabsdet = jnp.linalg.slogdet(vandermonde_matrix)
        F = sign * jnp.exp(logabsdet)
        return F  # should be a vector of shape (batch_size)

    def eta_symmetric(self, x: jax.Array):
        # x should be integers of shape (batch_size, n, 3) with entries from \widetilde{\Lambda}
        # contains (x, y, \sigma)
        # first flatten the last dimension with an injective map
        x = (
            ((2 * self.L) * (x[:, :, 0] % self.L))
            + (2 * (x[:, :, 1] % self.L))
            + x[:, :, 2]
        )
        N_s = (
            jax.nn.one_hot(x, num_classes=2 * (self.L**2)).sum(axis=1).astype("float32")
        )  # (batch_size, 2L^2)
        # take matrix-vector product with w_s
        eta = jnp.matmul(N_s, self.w_s)
        return eta  # should be a vector of shape (batch_size)

    def __call__(self, x: jax.Array):
        # x should be integers of shape (batch_size, n, 3) with entries from \widetilde{\Lambda}
        # contains (x, y, \sigma)
        # compute the antisymmetric function F
        F = self.F_antisymmetric(x)  # shape (batch_size), dtype=complex
        # compute the symmetric function g
        # first calculate eta
        eta = jnp.expand_dims(
            self.eta_symmetric(x), axis=-1
        )  # shape (batch_size, 1), dtype=float
        # now combine them together via:
        # \Psi = F_1 g_1 + F_2 g_2
        g_1 = view_as_complex(self.g_1_prime(eta))  # shape (batch_size), dtype=complex
        g_2 = view_as_complex(self.g_2_prime(eta))
        # the final result
        psi = (F.real * g_1) + (F.imag * g_2)  # shape (batch_size), dtype=complex
        return psi

### JAX impl of the gradient descent

In [12]:
# model parameters
L = 2
n = 2
diff = 0
t = 1
U = 2

hamiltonian = HamiltonianOperator(L, n, diff, t, U)
basis = jnp.asarray(hamiltonian.basis)
H_matrix = jnp.asarray(hamiltonian.H)

# instantiate the NN model & optimizer
model = psi_2D_spinful(L, hidden_layers=3, dim_feedforward=512, rngs=nnx.Rngs(0))
optimizer = nnx.Optimizer(model, optax.adamw(learning_rate=3e-4), wrt=nnx.Param)

# now implement the training loop
ED_energy = -7.570521831512  # you can put the value here
NN_energy = float("inf")  # the NN energy
error_criterion = 1e-7
epoch = 0
training_losses = []


def calculate_energy(model: nnx.Module, basis: jax.Array, H_matrix: jax.Array):
    # compute Psi
    Psi = model(basis)

    # the expectation value
    E = jnp.vdot(Psi, H_matrix @ Psi) / jnp.vdot(Psi, Psi)

    return E.real


@nnx.jit
def train_step(
    model: nnx.Module, optimizer: nnx.Optimizer, basis: jax.Array, H_matrix: jax.Array
):
    E, grads = nnx.value_and_grad(calculate_energy)(model, basis, H_matrix)

    # backpropagation
    optimizer.update(model, grads)
    return E


while NN_energy - ED_energy > error_criterion:
    E = train_step(model, optimizer, basis, H_matrix)

    # logging
    NN_energy = E.item()
    training_losses.append(NN_energy)
    print(f"Epoch {epoch}, E={NN_energy:.6f}")
    epoch += 1

W0310 10:56:47.167159 3293515 cpp_gen_intrinsics.cc:74] Empty bitcode string provided for eigen. Optimizations relying on this IR will be disabled.


Epoch 0, E=-1.250478
Epoch 1, E=-2.791028
Epoch 2, E=-2.908788
Epoch 3, E=-3.021099
Epoch 4, E=-3.115415
Epoch 5, E=-3.198282
Epoch 6, E=-3.279499
Epoch 7, E=-3.367592
Epoch 8, E=-3.471048
Epoch 9, E=-3.600277
Epoch 10, E=-3.770147
Epoch 11, E=-4.004037
Epoch 12, E=-4.339848
Epoch 13, E=-4.829598
Epoch 14, E=-5.457950
Epoch 15, E=-5.791762
Epoch 16, E=-5.476103
Epoch 17, E=-5.834387
Epoch 18, E=-6.084525
Epoch 19, E=-6.109612
Epoch 20, E=-6.081840
Epoch 21, E=-6.070384
Epoch 22, E=-6.092654
Epoch 23, E=-6.144370
Epoch 24, E=-6.207128
Epoch 25, E=-6.257954
Epoch 26, E=-6.286014
Epoch 27, E=-6.301327
Epoch 28, E=-6.316904
Epoch 29, E=-6.332871
Epoch 30, E=-6.347492
Epoch 31, E=-6.366793
Epoch 32, E=-6.394906
Epoch 33, E=-6.427186
Epoch 34, E=-6.455580
Epoch 35, E=-6.476390
Epoch 36, E=-6.493140
Epoch 37, E=-6.508867
Epoch 38, E=-6.521297
Epoch 39, E=-6.530951
Epoch 40, E=-6.543242
Epoch 41, E=-6.560908
Epoch 42, E=-6.581203
Epoch 43, E=-6.600242
Epoch 44, E=-6.617513
Epoch 45, E=-6.63464